In [ ]:

# Imports & Setup
import pandas as pd

# Load summarized cluster profile
df = spark.read.format("delta").table("gold.segments").toPandas()
cluster_profile = df.copy()
cluster_profile.index = [0, 1, 2, 3]  

# round for readability
cluster_profile = cluster_profile.round(2)

# round for readability
cluster_profile = cluster_profile.round(2)

# Generate Marketing Persona Using LLM
def generate_persona(row, cluster_id):
    """
    Uses a large language model to create a professional marketing persona
    for a given customer cluster.
    """
    # Example prompt sent to the model
    prompt = f"""
    You are a senior marketing strategist.
    Generate a realistic marketing persona for Cluster {cluster_id}.
    Data: <cluster statistics here>
    Requirements: 5-8 bullet points, business-ready persona
    """
    try:
        # Call the LLM API
        response = client.chat_completion(
            model="model_chosen",  # selected LLM
            messages=[
                {"role": "system", "content": "You are a senior marketing strategist."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=512,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Generate All Personas
dynamic_personas = {}

for cid in cluster_profile.index:
    row = cluster_profile.loc[cid]
    dynamic_personas[cid] = generate_persona(row, cid)

# Save to Delta Table
results_df = pd.DataFrame({
    "cluster_id": list(dynamic_personas.keys()),
    "persona":    list(dynamic_personas.values())
})

# Save LLM-generated personas
spark.createDataFrame(results_df) \
     .write.mode("overwrite") \
     .saveAsTable("gold.personas")

